In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_295K_278464_17O_opt_magres.magres')

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-51.76216646 165.84862613 -79.15415126]
 [172.92109001  54.32180818  28.48008084]
 [  8.98456411 -24.91510788 -46.19466476]]

17O2 sigma:
 [[ -51.76216646 -165.84862613   79.15415126]
 [-172.92109001   54.32180818   28.48008084]
 [  -8.98456411  -24.91510788  -46.19466476]]

17O3 sigma:
 [[-51.76216646 165.84862613  79.15415126]
 [172.92109001  54.32180818 -28.48008084]
 [ -8.98456411  24.91510788 -46.19466476]]

17O4 sigma:
 [[ -51.76216646 -165.84862613  -79.15415126]
 [-172.92109001   54.32180818  -28.48008084]
 [   8.98456411   24.91510788  -46.19466476]]

17O5 sigma:
 [[ -40.78847648  225.05282945   53.92102317]
 [ 194.61431093  106.55726755  -87.20255622]
 [  16.8554785   -49.92206346 -171.92816106]]

17O6 sigma:
 [[ -40.78847648 -225.05282945  -53.92102317]
 [-194.61431093  106.55726755  -87.20255622]
 [ -16.8554785   -49.92206346 -171.92816106]]

17O7 sigma:
 [[ -40.78847648  225.05282945  -53.92102317]
 [ 194.61431093  106.55726755   87.20255622]
 [ -16.8554785 

In [6]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.2791786921842325

17O2 sigma:
 6.279178692184241

17O3 sigma:
 6.279178692184209

17O4 sigma:
 6.2791786921842005

17O5 sigma:
 8.255981226978793

17O6 sigma:
 8.255981226978772

17O7 sigma:
 8.255981226978804

17O8 sigma:
 8.25598122697881



In [7]:
Q = -0.0256 #electric quadrupole moment for O17 in barn

CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + l = 1 + l = 2) Tensor from magres
CS_total[0,0] = -40.7885; CS_total[0,1] = 225.0528; CS_total[0,2] = 53.9210;
CS_total[1,0] = 194.6143; CS_total[1,1] = 106.5573; CS_total[1,2] = -87.2026;
CS_total[2,0] = 16.8555; CS_total[2,1] = -49.9221; CS_total[2,2] =   -171.9282;

Cs = np.zeros((3,3)) # CS symmetric (l = 0 + l = 2) 

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)
efg[0,0]= 0.6962; efg[0,1]= -0.1568; efg[0,2]=  0.1318;
efg[1,0]= efg[0,1]; efg[1,1]= 0.6385; efg[1,2]= -0.2574;
efg[2,0]= efg[0,2]; efg[2,1]= efg[1,2]; efg[2,2]= -1.3346;

# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 

V = efg*Q*234.9647

print(V)

print(Cs)

[[-4.18771006  0.9431671  -0.79278969]
 [ 0.9431671  -3.840639    1.54828579]
 [-0.79278969  1.54828579  8.02774755]]
[[ -40.7885   209.83355   35.38825]
 [ 209.83355  106.5573   -68.56235]
 [  35.38825  -68.56235 -171.9282 ]]


In [8]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [-5.17043202 -3.09248513  8.26231564] 

 Unsorted Eigenvectors:
 [[-0.73681608  0.67394782 -0.05381829]
 [ 0.66494623  0.73676351  0.12258072]
 [-0.12226436 -0.05453317  0.99099826]] 

Sorted Eigenvalues: 
 [-3.09248513 -5.17043202  8.26231564] 

Sorted Eigenvectors: 
 [[ 0.67394782 -0.73681608 -0.05381829]
 [ 0.73676351  0.66494623  0.12258072]
 [-0.05453317 -0.12226436  0.99099826]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 258.27388362 -113.54079682 -250.8924868 ] 

 Unsorted Eigenvectors:
 [[-0.56562484 -0.5862461  -0.57998625]
 [-0.82035154  0.32817601  0.46832025]
 [ 0.08421334 -0.74068618  0.6665524 ]] 

Sorted Eigenvalues: 
 [-113.54079682 -250.8924868   258.27388362] 

Sorted Eigenvectors: 
 [[-0.5862461  -0.57998625 -0.56562484]
 [ 0.32817601  0.46832025 -0.82035154]
 [-0.74068618  0.6665524   0.08421334]] 



In [9]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -3.0924851341627146 -5.170432017721238 8.262315642251957
CSA Tensor Components δyy, δxx, δzz: 
 -113.54079681583023 -250.89248680087374 258.27388361670404


In [10]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        8.26232
etaq            0.251497
iso_cs (ppm)  -35.3865
csa (ppm)     293.66
etas            0.467723


In [11]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.73681608  0.67394782 -0.05381829]
 [ 0.66494623  0.73676351  0.12258072]
 [-0.12226436 -0.05453317  0.99099826]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
24.03814942521155 7.693555583097455 66.29642705891291 

Direction cosine csa: 

[[-0.57998625 -0.5862461  -0.56562484]
 [ 0.46832025  0.32817601 -0.82035154]
 [ 0.6665524  -0.74068618  0.08421334]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-48.01557233381324 85.16920937048984 -55.414094042869145 



In [12]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -41.47542921253241 chi: 89.23582593188269 xi: -82.00788335003014 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[ -40.7885   209.83355   35.38825]
 [ 209.83355  106.5573   -68.56235]
 [  35.38825  -68.56235 -171.9282 ]]
CSA Tensor in Tenon Frame: 
 [[ 166.39861163 -142.52131396 -109.76229683]
 [-142.52131396 -201.58791163   47.22731623]
 [-109.76229683   47.22731623  -70.9701    ]]
Quad Tensor in Crystal Frame: 
 [[-4.18771006  0.9431671  -0.79278969]
 [ 0.9431671  -3.840639    1.54828579]
 [-0.79278969  1.54828579  8.02774755]]
Quad Tensor in Tenon Frame: 
 [[-3.78937317 -0.71862652 -0.67436273]
 [-0.71862652  2.66154261 -6.32085871]
 [-0.67436273 -6.32085871  1.12722905]]
